# 07 - CAF Trajectory Analysis (STORIES)

Analysis of Cancer-Associated Fibroblast (CAF) trajectories using the STORIES SpaceTime model.

This notebook performs:
- Spatial adjacency annotation (KNN-based CAF adjacent to tumor)
- Tumor bed annotation and patient filtering
- Dimensionality reduction (PCA, Isomap, UMAP)
- STORIES SpaceTime model fitting for trajectory inference
- Potential and velocity computation
- CellRank fate probability analysis (NR vs CR CAF)
- Spatial fate mapping
- DEG analysis (high vs low fate probability)
- Velocity and branch analysis
- Driver gene trends for NR and CR branches
- TF enrichment analysis

**Conda environment:** `stories-rsc-cuda126`

In [ ]:
import os

import anndata as ad
import cellrank as cr
import jax
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optax
import pandas as pd
import rapids_singlecell
import scanpy as sc
import seaborn as sns
import stories

from cellrank.kernels import VelocityKernel, ConnectivityKernel
from cellrank.estimators import GPCCA
from matplotlib.ticker import ScalarFormatter
from scipy import sparse
from scipy.stats import mannwhitneyu
from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

jax.devices()

## Configuration

In [ ]:
DATA_DIR = "../data"
RESULTS_DIR = "../results"
FIGURES_DIR = "../figures"
CHECKPOINT_DIR = os.path.join(RESULTS_DIR, "ckpt_caf_trajectory")

PATIENTS = ["Pt-7", "Pt-6", "Pt-4", "Pt-22", "Pt-24"]

DIST_THRESHOLD = 10  # um for spatial adjacency

# Leiden clusters to exclude
EXCLUDE_LEIDEN = [
    "CAF c10", "CAF c0", "CAF c11",
    "CAF c12", "CAF c13", "CAF c15", "CAF c16",
]

# CAF annotation mapping
NR_CAF_CLUSTERS = ["CAF c0", "CAF c3", "CAF c4"]
CR_CAF_CLUSTERS = ["CAF c1"]

# Plot style
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.linewidth"] = 1.2
sns.set_style("white")

## Data Loading

In [ ]:
# Load CAF and Tumor h5ad files
adata_caf = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_caf.h5ad"))
adata_tumor = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_tumor.h5ad"))

# Concatenate CAF + Tumor
adata = ad.concat([adata_caf, adata_tumor]).copy()
adata.obs_names_make_unique()

del adata_caf, adata_tumor

## Spatial Adjacency Annotation (KNN-based)

In [ ]:
adata.obs["CAF_adjacent_to_tumor"] = False

for sample in adata.obs["Sample"].unique():
    sub = adata[adata.obs["Sample"] == sample]

    caf_idx = np.where(sub.obs["Cell_type_integrate"] == "CAF")[0]
    tumor_idx = np.where(sub.obs["Cell_type_integrate"] == "Tumor")[0]

    if len(caf_idx) == 0 or len(tumor_idx) == 0:
        continue

    coords = sub.obs[["imagecol", "imagerow"]].values
    caf_coords = coords[caf_idx]
    tumor_coords = coords[tumor_idx]

    nbrs = NearestNeighbors(n_neighbors=1).fit(tumor_coords)
    dist, _ = nbrs.kneighbors(caf_coords)

    is_adjacent = (dist[:, 0] <= DIST_THRESHOLD)

    global_caf_idx = adata.obs.index.get_indexer(sub.obs.iloc[caf_idx].index)
    adata.obs.iloc[global_caf_idx, adata.obs.columns.get_loc("CAF_adjacent_to_tumor")] = is_adjacent

## Tumor Bed Annotation & Patient Filtering

In [ ]:
adata.obs["integrate_leiden"] = adata.obs["leiden"]

# Filter out unwanted leiden clusters
adata = adata[~adata.obs["integrate_leiden"].isin(EXCLUDE_LEIDEN)]

# Make index unique with sample info
adata.obs.index = adata.obs.index.astype(str) + "_" + adata.obs["Sample"].astype(str)

# Load tumor bed annotations
df_pt22 = pd.read_csv(os.path.join(DATA_DIR, "tumor_bed_pt22.csv"), comment="#")
df_pt22["Cell ID"] = df_pt22["Cell ID"] + "_Pt-22_Resection"

df_pt24 = pd.read_csv(os.path.join(DATA_DIR, "tumor_bed_pt24.csv"), comment="#")
df_pt24["Cell ID"] = df_pt24["Cell ID"] + "_Pt-24_Resection"

adata.obs["Tumor_bed"] = False
adata.obs.loc[adata.obs.index.isin(df_pt22["Cell ID"]), "Tumor_bed"] = True
adata.obs.loc[adata.obs.index.isin(df_pt24["Cell ID"]), "Tumor_bed"] = True

# Keep only CAF cells adjacent to tumor or in tumor bed
adata = adata[(adata.obs["CAF_adjacent_to_tumor"]) | (adata.obs["Tumor_bed"])]
adata = adata[adata.obs["Cell_type_integrate"] == "CAF"]

pd.crosstab(adata.obs["Sample"], adata.obs["Cell_type_integrate"]).head(10)

In [ ]:
# Filter to selected patients
adata = adata[adata.obs["Patient"].isin(PATIENTS)].copy()

# Annotate CAF subtypes
adata.obs["CAF_anno"] = "Other CAF"
adata.obs.loc[
    adata.obs["integrate_leiden"].isin(NR_CAF_CLUSTERS), "CAF_anno"
] = "NR CAF"
adata.obs.loc[
    adata.obs["integrate_leiden"].isin(CR_CAF_CLUSTERS), "CAF_anno"
] = "CR CAF"

# Remove excluded clusters again after patient filtering
adata = adata[~adata.obs["integrate_leiden"].isin(EXCLUDE_LEIDEN)]

## Dimensionality Reduction (PCA, Isomap, UMAP)

In [ ]:
adata_iso = adata.copy()

# PCA
sc.pp.scale(adata_iso)
sc.tl.pca(adata_iso, n_comps=50)

# Isomap on PCA coordinates
X_pca = adata_iso.obsm["X_pca"]
isomap = Isomap(n_neighbors=20, n_components=2)
X_iso = isomap.fit_transform(X_pca)
adata.obsm["X_isomap"] = X_iso

In [ ]:
sc.pl.umap(
    adata,
    color=["CAF_anno", "Timepoint"],
)

In [ ]:
sc.pl.scatter(
    adata,
    basis="isomap",
    color=["CAF_anno", "Timepoint", "POSTN", "integrate_leiden"],
    color_map="viridis",
    size=15,
)

In [ ]:
# Remove JustAfter timepoint
adata = adata[adata.obs["Timepoint"] != "JustAfter"]

# Select PCA components and normalize
adata.obsm["X_pca"] = adata.obsm["X_pca"][:, :20]
adata.obsm["X_pca"] /= adata.obsm["X_pca"].max()

# Assign time labels
adata.obs["time"] = adata.obs["Timepoint"].replace({
    "Pre": 0,
    "Resection": 1,
})

In [ ]:
# Shift spatial coordinates so samples are laid out side by side
samples = adata.obs["Sample"].unique().tolist()

sample_ranges = {}
for s in samples:
    df_s = adata[adata.obs["Sample"] == s]
    x = df_s.obs["imagecol"]
    y = df_s.obs["imagerow"]
    sample_ranges[s] = {
        "x_range": x.max() - x.min(),
        "y_range": y.max() - y.min(),
    }

x_offset = 0
OFFSET_MARGIN = 500

new_spatial = adata.obsm["spatial"].copy()
new_imagecol = adata.obs["imagecol"].copy()
new_imagerow = adata.obs["imagerow"].copy()

for s in samples:
    mask = adata.obs["Sample"] == s
    x_min = new_imagecol[mask].min()
    y_min = new_imagerow[mask].min()

    dx = x_offset - x_min
    dy = 0 - y_min

    new_imagecol[mask] += dx
    new_imagerow[mask] += dy
    new_spatial[mask, 0] += dx
    new_spatial[mask, 1] += dy

    x_offset += sample_ranges[s]["x_range"] + OFFSET_MARGIN

adata.obs["imagecol"] = new_imagecol
adata.obs["imagerow"] = new_imagerow
adata.obsm["spatial"] = new_spatial

In [ ]:
# Center and scale each timepoint in space
timepoints = np.sort(np.unique(adata.obs["time"]))
adata.obsm["spatial"] = adata.obsm["spatial"].astype(float)
for t in timepoints:
    idx = adata.obs["time"] == t
    mu = np.mean(adata.obsm["spatial"][idx, :], axis=0)
    adata.obsm["spatial"][idx, :] -= mu
    std = np.std(adata.obsm["spatial"][idx, :], axis=0)
    adata.obsm["spatial"][idx, :] /= std

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(10, 5), sharey=True, sharex=True, constrained_layout=True
)
for i, t in enumerate(sorted(adata.obs["time"].unique())):
    idx = adata.obs["time"] == t
    sc.pl.embedding(
        adata[idx], "spatial", color="CAF_anno", s=30, ax=axes[i], show=False
    )

## STORIES SpaceTime Model Fitting

In [ ]:
# Initialize the model
model = stories.SpaceTime(quadratic_weight=1e-3)

adata.obs["time"] = adata.obs["time"].astype(int)

scheduler = optax.cosine_decay_schedule(1e-2, 10_000)
model.fit(
    adata=adata,
    time_key="time",
    omics_key="X_pca",
    space_key="spatial",
    optimizer=optax.adamw(scheduler),
    checkpoint_manager=CHECKPOINT_DIR,
)

## Potential & Velocity Computation

In [ ]:
stories.tools.compute_potential(adata, model, "X_pca")

In [ ]:
sc.pl.embedding(
    adata,
    basis="X_isomap",
    color=["CAF_anno", "potential", "Timepoint", "POSTN", "leiden"],
    vmax="p98",
)

In [ ]:
# Save potential with isomap coordinates
potential_fn = lambda x: model.potential.apply(model.params, x)
df_potential = pd.DataFrame(np.array(potential_fn(adata.obsm["X_pca"])))
df_potential[["iso_1", "iso_2"]] = adata.obsm["X_isomap"]
df_potential = df_potential.rename(columns={0: "potential"})
df_potential = pd.concat([df_potential, adata.obs[["Timepoint"]].reset_index(drop=True)], axis=1)
df_potential.to_csv(os.path.join(RESULTS_DIR, "potential_with_isomap.csv"))

In [ ]:
sc.pl.embedding(
    adata,
    basis="umap",
    color=["potential", "POSTN"],
    vmax="p98",
)

### Custom Velocity Computation

In [ ]:
def compute_velocity(
    adata,
    model,
    omics_key,
    key_added="X_velo",
    save_transition=True,
    transition_key="T_fwd",
    backend="threading",
):
    """Compute -grad J for all cells and optionally save CellRank transition matrix.

    This overrides the library version to provide more control over the
    transition matrix computation.

    Parameters
    ----------
    adata : AnnData
        Input data.
    model : stories.SpaceTime
        Trained model.
    omics_key : str
        obsm key for omics representation (e.g. 'X_pca').
    key_added : str
        obsm key to store velocity.
    save_transition : bool
        Whether to compute/save forward transition matrix.
    transition_key : str
        Key in obsp to store transition matrix.
    backend : str
        Backend for CellRank compute_transition_matrix.
    """
    potential_fn = lambda x: model.potential.apply(model.params, x)
    velo_fn = lambda x: -jax.vmap(jax.grad(potential_fn))(x)

    # Save velocity vectors
    adata.obsm[key_added] = np.array(velo_fn(adata.obsm[omics_key]))

    # Optionally compute/save transition matrix
    if save_transition:
        vk = cr.kernels.VelocityKernel(
            adata,
            attr="obsm",
            xkey=omics_key,
            vkey=key_added,
        ).compute_transition_matrix(backend=backend)

        adata.obsp[transition_key] = vk.transition_matrix.copy()
        adata.uns[f"{transition_key}_params"] = {
            "kernel": "VelocityKernel",
            "attr": "obsm",
            "xkey": omics_key,
            "vkey": key_added,
            "backend": backend,
        }

In [ ]:
# Compute neighbors for velocity plotting
sc.pp.neighbors(
    adata,
    n_neighbors=30,
    n_pcs=None,
    use_rep="X_pca",
)

# Run library velocity first (for comparison)
stories.tools.compute_velocity(adata, model, "X_pca")

# Override with custom velocity computation
compute_velocity(
    adata,
    model,
    "X_pca",
    key_added="X_velo",
    save_transition=True,
    transition_key="T_fwd",
)

## CellRank Fate Probability Analysis

In [ ]:
k = cr.kernels.PrecomputedKernel(adata.obsp["T_fwd"])
g = cr.estimators.GPCCA(k)

ct_key = "CAF_anno"
n_states = adata.obs[ct_key].nunique()

g.compute_schur(n_components=min(30, n_states + 10))
g.compute_macrostates(n_states=n_states)

In [ ]:
M = g.macrostates_memberships

memberships = pd.DataFrame(
    np.asarray(M.X),
    index=adata.obs_names,
    columns=[str(x) for x in M.names],
)

macro = memberships.idxmax(axis=1)
anno = adata.obs[ct_key].astype(str).str.strip()
tab = pd.crosstab(macro, anno)
print(tab)

In [ ]:
# Set NR CAF terminal state and compute fate probabilities
nr_macro = str(tab["NR CAF"].idxmax())
print("NR macrostate:", nr_macro)

g.set_terminal_states(states=[nr_macro])
g.compute_fate_probabilities()
adata.obs["fate_prob_NR_CAF"] = np.asarray(g.fate_probabilities.X).ravel()

# Set CR CAF terminal state and compute fate probabilities
cr_macro = str(tab["CR CAF"].idxmax())
print("CR macrostate:", cr_macro)

g.set_terminal_states(states=[cr_macro])
g.compute_fate_probabilities()
adata.obs["fate_prob_CR_CAF"] = np.asarray(g.fate_probabilities.X).ravel()

sc.pl.umap(adata, color=["CAF_anno", "fate_prob_NR_CAF", "fate_prob_CR_CAF"])

In [ ]:
sc.pl.scatter(
    adata,
    basis="isomap",
    color=["CAF_anno", "fate_prob_NR_CAF", "fate_prob_CR_CAF"],
    color_map="viridis",
    size=15,
)

## Spatial Fate Mapping

In [ ]:
# Plot Pre-treatment CAF fate probabilities in spatial coordinates
adata_plot = adata.copy()

mask = (
    (adata_plot.obs["Timepoint"].astype(str).str.strip() == "Pre")
    & (adata_plot.obs["CAF_anno"].astype(str).str.strip().isin(["Other CAF", "NR CAF", "CR CAF"]))
)
adata_pre_caf = adata_plot[mask].copy()

if "spatial" in adata_pre_caf.obsm:
    coords = adata_pre_caf.obsm["spatial"]
    x = coords[:, 0]
    y = coords[:, 1]
else:
    x = adata_pre_caf.obs["imagecol"].values
    y = adata_pre_caf.obs["imagerow"].values

p_nr = adata_pre_caf.obs["fate_prob_NR_CAF"].astype(float).values
p_cr = adata_pre_caf.obs["fate_prob_CR_CAF"].astype(float).values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc1 = axes[0].scatter(x, y, c=p_nr, s=8, cmap="Blues", linewidths=0)
axes[0].set_title("Pre CAF: fate probability to NR CAF", fontsize=14)
axes[0].set_xlabel("imagecol")
axes[0].set_ylabel("imagerow")
axes[0].invert_yaxis()
plt.colorbar(sc1, ax=axes[0], fraction=0.046, pad=0.04)

sc2 = axes[1].scatter(x, y, c=p_cr, s=8, cmap="Blues", linewidths=0)
axes[1].set_title("Pre CAF: fate probability to CR CAF", fontsize=14)
axes[1].set_xlabel("imagecol")
axes[1].set_ylabel("imagerow")
axes[1].invert_yaxis()
plt.colorbar(sc2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
obs = adata.obs.copy()
obs["Timepoint"] = obs["Timepoint"].astype(str).str.strip()
obs["CAF_anno"] = obs["CAF_anno"].astype(str).str.strip()
obs["Response"] = obs["Response"].astype(str).str.strip()

# Use NR instead of SD
obs["Response"] = obs["Response"].replace({"SD": "NR"})

plot_df = obs[
    (obs["Timepoint"] == "Pre")
    & (obs["CAF_anno"].isin(["Other CAF"]))
    & (obs["Response"].isin(["NR", "CR"]))
].copy()

plot_df["fate_prob_NR_CAF"] = pd.to_numeric(plot_df["fate_prob_NR_CAF"], errors="coerce")
plot_df["fate_prob_CR_CAF"] = pd.to_numeric(plot_df["fate_prob_CR_CAF"], errors="coerce")
plot_df = plot_df.dropna(subset=["fate_prob_NR_CAF", "fate_prob_CR_CAF"])

# Statistics
nr1 = plot_df.loc[plot_df["Response"] == "NR", "fate_prob_NR_CAF"]
cr1 = plot_df.loc[plot_df["Response"] == "CR", "fate_prob_NR_CAF"]
p1 = mannwhitneyu(nr1, cr1, alternative="two-sided").pvalue

nr2 = plot_df.loc[plot_df["Response"] == "NR", "fate_prob_CR_CAF"]
cr2 = plot_df.loc[plot_df["Response"] == "CR", "fate_prob_CR_CAF"]
p2 = mannwhitneyu(nr2, cr2, alternative="two-sided").pvalue


def p_to_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


palette = {
    "NR": "#d95f5f",
    "CR": "#4c78a8",
}


def add_sig_annotation(ax, data, y, order=("NR", "CR"), pval=1.0, line_color="black"):
    y_max = data[y].max()
    y_min = data[y].min()
    yr = y_max - y_min
    if yr == 0:
        yr = 0.1

    bar_y = y_max + 0.08 * yr
    bar_h = 0.04 * yr
    text_y = bar_y + bar_h + 0.02 * yr

    x1, x2 = 0, 1
    ax.plot(
        [x1, x1, x2, x2],
        [bar_y, bar_y + bar_h, bar_y + bar_h, bar_y],
        lw=1.3,
        c=line_color,
        clip_on=False,
    )

    ax.text(
        (x1 + x2) / 2,
        text_y,
        f"{p_to_stars(pval)}\nP = {pval:.2e}",
        ha="center",
        va="bottom",
        fontsize=11,
    )

    ax.set_ylim(y_min - 0.05 * yr, y_max + 0.28 * yr)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(4.8, 8.2), dpi=300)

# Top: fate to NR CAF
ax = axes[0]
sns.violinplot(
    data=plot_df, x="Response", y="fate_prob_NR_CAF",
    order=["NR", "CR"], palette=palette,
    inner=None, cut=0, linewidth=1.2, saturation=1, ax=ax,
)
sns.boxplot(
    data=plot_df, x="Response", y="fate_prob_NR_CAF",
    order=["NR", "CR"], width=0.22, showcaps=True,
    boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.1, "zorder": 3},
    whiskerprops={"color": "black", "linewidth": 1.1},
    capprops={"color": "black", "linewidth": 1.1},
    medianprops={"color": "black", "linewidth": 1.3},
    showfliers=False, ax=ax,
)
sns.stripplot(
    data=plot_df, x="Response", y="fate_prob_NR_CAF",
    order=["NR", "CR"], color="black",
    size=2.2, alpha=0.35, jitter=0.22, ax=ax,
)
ax.set_title("Fate probability to NR CAF", fontsize=13, pad=10)
ax.set_xlabel("")
ax.set_ylabel("Fate probability", fontsize=12)
add_sig_annotation(ax, plot_df, "fate_prob_NR_CAF", pval=p1)

# Bottom: fate to CR CAF
ax = axes[1]
sns.violinplot(
    data=plot_df, x="Response", y="fate_prob_CR_CAF",
    order=["NR", "CR"], palette=palette,
    inner=None, cut=0, linewidth=1.2, saturation=1, ax=ax,
)
sns.boxplot(
    data=plot_df, x="Response", y="fate_prob_CR_CAF",
    order=["NR", "CR"], width=0.22, showcaps=True,
    boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.1, "zorder": 3},
    whiskerprops={"color": "black", "linewidth": 1.1},
    capprops={"color": "black", "linewidth": 1.1},
    medianprops={"color": "black", "linewidth": 1.3},
    showfliers=False, ax=ax,
)
sns.stripplot(
    data=plot_df, x="Response", y="fate_prob_CR_CAF",
    order=["NR", "CR"], color="black",
    size=2.2, alpha=0.35, jitter=0.22, ax=ax,
)
ax.set_title("Fate probability to CR CAF", fontsize=13, pad=10)
ax.set_xlabel("")
ax.set_ylabel("Fate probability", fontsize=12)
add_sig_annotation(ax, plot_df, "fate_prob_CR_CAF", pval=p2)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=11)
    ax.grid(False)

fig.suptitle("Pre-treatment Other CAF", fontsize=14, y=0.995)
plt.tight_layout()
fig.subplots_adjust(hspace=0.35)
plt.show()

In [ ]:
obs.to_csv(os.path.join(RESULTS_DIR, "fate_prob.tsv"), sep="\t")

### Spatial Fate Mapping with Full Data Context

In [ ]:
# Reload full data for spatial fate visualization
adata_full = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_caf.h5ad"))
adata_full_tumor = ad.read_h5ad(os.path.join(DATA_DIR, "integrate_adata_filtered_tumor.h5ad"))
adata_full = ad.concat([adata_full, adata_full_tumor]).copy()
del adata_full_tumor

# Match indices
adata_full.obs.index = adata_full.obs.index + "_" + adata_full.obs["Sample"].astype(str)
adata_full.obs.index = adata_full.obs.index.str.lower()

# Load and merge fate probabilities
obs_fate = pd.read_table(os.path.join(RESULTS_DIR, "fate_prob.tsv"), index_col=0)
obs_fate.index = obs_fate.index.str.lower()

adata_full.obs = pd.merge(
    adata_full.obs,
    obs_fate[["fate_prob_CR_CAF", "fate_prob_NR_CAF"]],
    right_index=True,
    left_index=True,
    how="left",
)

# Annotate CAF subtypes on full data
adata_full = adata_full[adata_full.obs["Patient"].isin(PATIENTS)].copy()
adata_full.obs["integrate_leiden"] = adata_full.obs["leiden"]
adata_full.obs["CAF_anno"] = "Other CAF"
adata_full.obs.loc[
    adata_full.obs["integrate_leiden"].isin(NR_CAF_CLUSTERS), "CAF_anno"
] = "NR CAF"
adata_full.obs.loc[
    adata_full.obs["integrate_leiden"].isin(CR_CAF_CLUSTERS), "CAF_anno"
] = "CR CAF"
adata_full = adata_full[~adata_full.obs["integrate_leiden"].isin(EXCLUDE_LEIDEN)]

In [ ]:
def add_scalebar(ax, length_um=200, pixel_size=0.2125):
    length_px = length_um / pixel_size
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    x = x1 - (x1 - x0) * 0.05 - length_px
    y = y0 + (y1 - y0) * 0.05
    ax.plot([x, x + length_px], [y, y], color="black", linewidth=3)
    ax.text(
        x + length_px / 2,
        y + (y1 - y0) * 0.03,
        f"{length_um} um",
        ha="center",
        va="bottom",
        fontsize=9,
    )

In [ ]:
obs_plot = adata_full.obs.copy()
plot_df = obs_plot[
    (obs_plot["Patient"].isin(PATIENTS))
    & (obs_plot["Timepoint"] == "Pre")
].copy()

vmin = plot_df["fate_prob_NR_CAF"].min()
vmax = plot_df["fate_prob_NR_CAF"].max()

centers = {}
widths = []
heights = []

for pt in PATIENTS:
    df_pt = plot_df[plot_df["Patient"] == pt]
    xmin_pt, xmax_pt = df_pt["imagecol"].min(), df_pt["imagecol"].max()
    ymin_pt, ymax_pt = df_pt["imagerow"].min(), df_pt["imagerow"].max()
    centers[pt] = ((xmin_pt + xmax_pt) / 2, (ymin_pt + ymax_pt) / 2)
    widths.append(xmax_pt - xmin_pt)
    heights.append(ymax_pt - ymin_pt)

global_range = max(max(widths), max(heights))
half_x = global_range / 2
half_y = global_range / 2

fig, axes = plt.subplots(1, len(PATIENTS), figsize=(14, 3.2))
scat = None

for ax, pt in zip(axes, PATIENTS):
    df = plot_df[plot_df["Patient"] == pt].copy()

    nan_df = df[df["fate_prob_NR_CAF"].isna()]
    ax.scatter(nan_df["imagecol"], nan_df["imagerow"], s=5, c="lightgray", linewidths=0)

    val_df = df[df["fate_prob_NR_CAF"].notna()]
    scat = ax.scatter(
        val_df["imagecol"], val_df["imagerow"],
        c=val_df["fate_prob_NR_CAF"], cmap="Reds",
        vmin=vmin, vmax=vmax, s=5, linewidths=0,
    )

    xcenter, ycenter = centers[pt]
    ax.set_xlim(xcenter - half_x, xcenter + half_x)
    ax.set_ylim(ycenter - half_y, ycenter + half_y)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(pt)
    add_scalebar(ax, length_um=200)

cbar = fig.colorbar(scat, ax=axes, fraction=0.015, pad=0.005)
cbar.set_label("fate_prob_NR_CAF")
formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
cbar.ax.yaxis.set_major_formatter(formatter)
cbar.update_ticks()

fig.subplots_adjust(left=0.01, right=0.92, bottom=0.02, top=0.90, wspace=-0.5)
plt.show()

In [ ]:
vmin_cr = plot_df["fate_prob_CR_CAF"].min()
vmax_cr = plot_df["fate_prob_CR_CAF"].max()

fig, axes = plt.subplots(1, len(PATIENTS), figsize=(14, 3.2))
scat = None

for ax, pt in zip(axes, PATIENTS):
    df = plot_df[plot_df["Patient"] == pt].copy()

    nan_df = df[df["fate_prob_CR_CAF"].isna()]
    ax.scatter(
        nan_df["imagecol"], nan_df["imagerow"],
        s=5, alpha=0.1, c="lightgray", linewidths=0,
    )

    val_df = df[df["fate_prob_CR_CAF"].notna()]
    scat = ax.scatter(
        val_df["imagecol"], val_df["imagerow"],
        c=val_df["fate_prob_CR_CAF"], cmap="Blues",
        vmin=vmin_cr, vmax=vmax_cr, s=5, linewidths=0,
    )

    xcenter, ycenter = centers[pt]
    ax.set_xlim(xcenter - half_x, xcenter + half_x)
    ax.set_ylim(ycenter - half_y, ycenter + half_y)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(pt)

cbar = fig.colorbar(scat, ax=axes, fraction=0.015, pad=0.005)
cbar.set_label("fate_prob_CR_CAF")
formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
cbar.ax.yaxis.set_major_formatter(formatter)
cbar.update_ticks()

fig.subplots_adjust(left=0.01, right=0.92, bottom=0.02, top=0.90, wspace=-0.5)
plt.show()

## DEG Analysis (High vs Low Fate Probability)

In [ ]:
adata_in = adata_full.copy()

fate_col = "fate_prob_NR_CAF"
group_col = "NR_fate_group"
group_high = "High"
group_low = "Low"

timepoint_col = "Timepoint"
cafanno_col = "CAF_anno"
target_timepoint = "Pre"
target_cafanno = "Other CAF"

outdir = os.path.join(RESULTS_DIR, "DEG_Pre_OtherCAF_NR_fate_high_vs_low")
os.makedirs(outdir, exist_ok=True)

pval_threshold = 0.05
logfc_threshold = 1
method = "wilcoxon"

# Filter to Pre + Other CAF
adata_in.obs[timepoint_col] = adata_in.obs[timepoint_col].astype(str).str.strip()
adata_in.obs[cafanno_col] = adata_in.obs[cafanno_col].astype(str).str.strip()

adata_deg = adata_in[
    (adata_in.obs[timepoint_col] == target_timepoint)
    & (adata_in.obs[cafanno_col] == target_cafanno)
].copy()

print("After Pre + Other CAF filtering:", adata_deg.shape)

# Numeric conversion and NaN removal
adata_deg.obs[fate_col] = pd.to_numeric(adata_deg.obs[fate_col], errors="coerce")
adata_deg = adata_deg[~adata_deg.obs[fate_col].isna()].copy()

# Top 25% vs bottom 25%
q25 = adata_deg.obs[fate_col].quantile(0.25)
q75 = adata_deg.obs[fate_col].quantile(0.75)

adata_deg = adata_deg[
    (adata_deg.obs[fate_col] <= q25) | (adata_deg.obs[fate_col] >= q75)
].copy()

adata_deg.obs[group_col] = np.where(
    adata_deg.obs[fate_col] >= q75, group_high, group_low
)
adata_deg.obs[group_col] = pd.Categorical(
    adata_deg.obs[group_col], categories=[group_low, group_high]
)

print("After quartile split:", adata_deg.shape)
print(adata_deg.obs[group_col].value_counts())

# DEG
use_raw = adata_deg.raw is not None
sc.tl.rank_genes_groups(
    adata_deg,
    groupby=group_col,
    groups=[group_high],
    reference=group_low,
    method=method,
    use_raw=use_raw,
    pts=True,
)

deg = sc.get.rank_genes_groups_df(adata_deg, group=group_high)
deg.to_csv(os.path.join(outdir, "DEG_all_High_vs_Low.csv"), index=False)

deg_sig_up = deg[
    (deg["pvals_adj"] < pval_threshold) & (deg["logfoldchanges"] > logfc_threshold)
].copy()
deg_sig_down = deg[
    (deg["pvals_adj"] < pval_threshold) & (deg["logfoldchanges"] < -logfc_threshold)
].copy()

deg_sig_up.to_csv(os.path.join(outdir, "DEG_sig_up_High_vs_Low.csv"), index=False)
deg_sig_down.to_csv(os.path.join(outdir, "DEG_sig_down_High_vs_Low.csv"), index=False)

print(f"Significant up in {group_high}: {deg_sig_up.shape[0]}")
print(f"Significant down in {group_high}: {deg_sig_down.shape[0]}")

In [ ]:
# Volcano plot
volcano_df = deg.copy()
volcano_df["neglog10_padj"] = -np.log10(volcano_df["pvals_adj"].clip(lower=1e-300))

plt.figure(figsize=(6, 5))
plt.scatter(volcano_df["logfoldchanges"], volcano_df["neglog10_padj"], s=8, alpha=0.5)

sig_mask_up = (volcano_df["pvals_adj"] < pval_threshold) & (volcano_df["logfoldchanges"] > logfc_threshold)
sig_mask_down = (volcano_df["pvals_adj"] < pval_threshold) & (volcano_df["logfoldchanges"] < -logfc_threshold)

plt.scatter(
    volcano_df.loc[sig_mask_up, "logfoldchanges"],
    volcano_df.loc[sig_mask_up, "neglog10_padj"],
    s=10, alpha=0.9, label=f"Up in {group_high}",
)
plt.scatter(
    volcano_df.loc[sig_mask_down, "logfoldchanges"],
    volcano_df.loc[sig_mask_down, "neglog10_padj"],
    s=10, alpha=0.9, label=f"Up in {group_low}",
)

plt.axvline(logfc_threshold, linestyle="--", linewidth=1)
plt.axvline(-logfc_threshold, linestyle="--", linewidth=1)
plt.axhline(-np.log10(pval_threshold), linestyle="--", linewidth=1)

top_label_n = 12
label_df = volcano_df[sig_mask_up | sig_mask_down].copy()
label_df = label_df.sort_values("pvals_adj", ascending=True).head(top_label_n)
for _, row in label_df.iterrows():
    plt.text(row["logfoldchanges"], row["neglog10_padj"], row["names"], fontsize=8)

plt.xlabel("log fold change")
plt.ylabel("-log10 adjusted p-value")
plt.title(f"{target_timepoint} {target_cafanno}: {group_high} vs {group_low}")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(outdir, "volcano_High_vs_Low.png"), dpi=300, bbox_inches="tight")
plt.show()

## Velocity & Branch Analysis

In [ ]:
palette_tp = {
    "Pre": (0.23, 0.49, 0.77),
    "Resection": (0.89, 0.47, 0.20),
}

stories.tools.plot_velocity(
    adata, "X_pca",
    basis="isomap",
    color="Timepoint",
    palette=palette_tp,
    s=50,
)

In [ ]:
adata.obs["Timepoint_Response"] = (
    adata.obs["Timepoint"].astype(str) + "_" + adata.obs["Response"].astype(str)
)
adata.obs["Timepoint_Response"] = [
    i.replace("SD", "NR") for i in adata.obs["Timepoint_Response"]
]

palette_tr = {
    "Pre_NR":       (0.20, 0.70, 0.30),
    "Pre_CR":       (0.60, 0.30, 0.80),
    "Resection_NR": (0.90, 0.15, 0.15),
    "Resection_CR": (0.15, 0.35, 0.90),
}

stories.tools.plot_velocity(
    adata, "X_pca",
    basis="isomap",
    color="Timepoint_Response",
    alpha=0.2,
    palette=palette_tr,
    s=50,
)

In [ ]:
# Save potential with Timepoint_Response annotation
potential_fn = lambda x: model.potential.apply(model.params, x)
df_potential_tr = pd.DataFrame(np.array(potential_fn(adata.obsm["X_pca"])))
df_potential_tr[["iso_1", "iso_2"]] = adata.obsm["X_isomap"]
df_potential_tr = df_potential_tr.rename(columns={0: "potential"})
df_potential_tr = pd.concat(
    [df_potential_tr, adata.obs[["Timepoint_Response"]].reset_index(drop=True)],
    axis=1,
)
df_potential_tr.to_csv(os.path.join(RESULTS_DIR, "potential_timepoint_response.csv"))

In [ ]:
# Plot velocity colored by CAF annotation
order = ["NR CAF", "Other CAF", "CR CAF"]
adata.obs["CAF_anno"] = pd.Categorical(
    adata.obs["CAF_anno"], categories=order, ordered=True
)
adata_sorted = adata[adata.obs["CAF_anno"].sort_values().index, :].copy()

stories.tools.plot_velocity(
    adata_sorted, "X_pca",
    basis="isomap",
    color="CAF_anno",
    s=50,
)

### Branch Assignment via CellRank

In [ ]:
# Prepare dense matrix for STORIES gene regression
adata_stories = adata.copy()
if not isinstance(adata_stories.X, np.ndarray):
    adata_stories.X = adata_stories.X.toarray()

In [ ]:
# Leiden clustering on stories data
rapids_singlecell.get.anndata_to_GPU(adata_stories)
rapids_singlecell.tl.leiden(adata_stories, random_state=0, resolution=1)
rapids_singlecell.get.anndata_to_CPU(adata_stories)

In [ ]:
# Clean leiden categories
leiden_cat = adata_stories.obs["leiden"].astype("category")
leiden_cat = leiden_cat.cat.remove_unused_categories()
adata_stories.obs["leiden"] = leiden_cat

if "leiden_colors" in adata_stories.uns:
    del adata_stories.uns["leiden_colors"]

sc.pl.scatter(
    adata_stories,
    basis="isomap",
    color=["leiden"],
    size=15,
)

In [ ]:
# CellRank branch analysis
sc.pp.neighbors(adata_stories, use_rep="X_pca")

vk = VelocityKernel(adata_stories, vkey="X_velo", xkey="X_pca", attr="obsm")
ck = ConnectivityKernel(adata_stories)
vk.compute_transition_matrix()
ck.compute_transition_matrix()
kernel = 0.8 * vk + 0.2 * ck

g = GPCCA(kernel)
g.fit(cluster_key="leiden", n_states=[4, 12])
g.predict_terminal_states()
g.predict_initial_states()
g.compute_fate_probabilities()

In [ ]:
# Identify NR and CR lineages
fate_df = g.fate_probabilities.X
fate_cols = g.fate_probabilities.names
fate_probs = pd.DataFrame(fate_df, index=adata_stories.obs_names, columns=fate_cols)

resp = adata_stories.obs["Response"]
lin_means = fate_probs.groupby(resp).mean()
print(lin_means)

nr_lin = lin_means.loc["SD"].idxmax()
cr_lin = lin_means.loc["CR"].idxmax()
print("NR lineage:", nr_lin)
print("CR lineage:", cr_lin)

In [ ]:
# Assign branches
adata_stories.obs["branch"] = "other"
adata_stories.obs.loc[fate_probs[nr_lin] > 0.5, "branch"] = "NR_branch"
adata_stories.obs.loc[fate_probs[cr_lin] > 0.5, "branch"] = "CR_branch"
adata_stories.obs["branch"] = adata_stories.obs["branch"].astype("category")
adata_stories.obs["branch"] = adata_stories.obs["branch"].cat.set_categories(
    ["NR_branch", "CR_branch", "other"]
)

# Pseudotime from potential
pt = -adata_stories.obs["potential"].copy()
pt = (pt - pt.min()) / (pt.max() - pt.min())
adata_stories.obs["stories_pt"] = pt

# Filter to branch cells only
adata_stories = adata_stories[adata_stories.obs["branch"] != "other"]

adata_stories.obs["branch"] = pd.Categorical(
    adata_stories.obs["branch"],
    categories=["NR_branch", "CR_branch"],
    ordered=True,
)

In [ ]:
# Plot velocity by branch
adata_sorted = adata_stories[adata_stories.obs["branch"].sort_values().index, :].copy()

stories.tools.plot_velocity(
    adata_sorted, "X_pca",
    basis="isomap",
    color="branch",
    s=50,
)

In [ ]:
# Branch composition plot
order_caf = ["CR CAF", "NR CAF", "Other CAF"]
adata_stories.obs["CAF_anno"] = pd.Categorical(
    adata_stories.obs["CAF_anno"], categories=order_caf, ordered=True
)

obs_comp = adata_stories.obs[["branch", "CAF_anno"]].copy()
comp = (
    obs_comp.groupby(["branch", "CAF_anno"]).size().reset_index(name="n_cells")
)
comp["frac"] = comp.groupby("branch")["n_cells"].transform(lambda x: x / x.sum())

pivot = comp.pivot(index="branch", columns="CAF_anno", values="frac").fillna(0)
branch_order = [b for b in ["NR_branch", "CR_branch"] if b in pivot.index]
if branch_order:
    pivot = pivot.loc[branch_order]

ax = pivot.plot(kind="bar", stacked=True, figsize=(4, 4))
ax.set_xlabel("")
ax.set_ylabel("Proportion of CAF cells")
ax.legend(title="CAF_anno", bbox_to_anchor=(1.05, 1.0), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Re-cluster branch data for visualization
rapids_singlecell.get.anndata_to_GPU(adata_stories)
rapids_singlecell.pp.neighbors(adata_stories, random_state=0, use_rep="X_isomap")
rapids_singlecell.tl.leiden(adata_stories, random_state=0, resolution=1.5)
rapids_singlecell.get.anndata_to_CPU(adata_stories)

leiden_cat = adata_stories.obs["leiden"].astype("category")
leiden_cat = leiden_cat.cat.remove_unused_categories()
adata_stories.obs["leiden"] = leiden_cat

if "leiden_colors" in adata_stories.uns:
    del adata_stories.uns["leiden_colors"]

sc.pl.scatter(
    adata_stories,
    basis="isomap",
    color=["leiden"],
    size=15,
)

In [ ]:
sc.pl.scatter(
    adata_stories,
    basis="isomap",
    color=["potential", "WNT5A", "POSTN", "SFRP1", "CXCL12"],
    size=15,
)

## Driver Gene Trends (NR and CR Branches)

In [ ]:
# Split into NR and CR branches
adata_stories_nr = adata_stories[adata_stories.obs["branch"] == "NR_branch"].copy()
adata_stories_cr = adata_stories[adata_stories.obs["branch"] == "CR_branch"].copy()

In [ ]:
# NR branch gene regression and driver genes
stories.tools.regress_genes(adata_stories_nr)

sorted_genes_names = stories.tools.select_driver_genes(
    adata_stories_nr, n_stages=7, n_genes=5,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_stories_nr, sorted_genes_names, title="NR branch",
)
plt.tight_layout()
plt.show()

In [ ]:
sorted_genes_names = stories.tools.select_driver_genes(
    adata_stories_nr, n_stages=2, n_genes=10,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_stories_nr, sorted_genes_names,
)
fig.set_size_inches(7, 6)
plt.tight_layout()
plt.show()

In [ ]:
# CR branch gene regression and driver genes
stories.tools.regress_genes(adata_stories_cr)

sorted_genes_names = stories.tools.select_driver_genes(
    adata_stories_cr, n_stages=2, n_genes=10,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_stories_cr, sorted_genes_names, title="CR branch",
)
plt.tight_layout()
plt.show()

In [ ]:
sorted_genes_names = stories.tools.select_driver_genes(
    adata_stories_cr, n_stages=4, n_genes=5,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_stories_cr, sorted_genes_names,
)
fig.set_size_inches(7, 6)
plt.tight_layout()
plt.show()

### Potential-Filtered Driver Gene Trends

In [ ]:
# NR branch: filter by potential percentile
x_nr = adata_stories_nr.obs["potential"].values
low_nr = np.percentile(x_nr, 0)
high_nr = np.percentile(x_nr, 80)

mask_nr = (adata_stories_nr.obs["potential"] >= low_nr) & (adata_stories_nr.obs["potential"] <= high_nr)
adata_nr_filtered = adata_stories_nr[mask_nr].copy()

stories.tools.regress_genes(adata_nr_filtered)

sorted_genes_names = stories.tools.select_driver_genes(
    adata_nr_filtered, n_stages=2, n_genes=10,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_nr_filtered, sorted_genes_names, title="NR branch (filtered)",
)
plt.tight_layout()
plt.show()

sc.pl.scatter(adata_nr_filtered, basis="isomap", color=["potential"], size=15)

In [ ]:
# CR branch: filter by potential percentile
x_cr = adata_stories_cr.obs["potential"].values
low_cr = np.percentile(x_cr, 0)
high_cr = np.percentile(x_cr, 80)

mask_cr = (adata_stories_cr.obs["potential"] >= low_cr) & (adata_stories_cr.obs["potential"] <= high_cr)
adata_cr_filtered = adata_stories_cr[mask_cr].copy()

stories.tools.regress_genes(adata_cr_filtered)

sorted_genes_names = stories.tools.select_driver_genes(
    adata_cr_filtered, n_stages=2, n_genes=10,
)
fig, axes = stories.tools.plot_gene_trends(
    adata_cr_filtered, sorted_genes_names, title="CR branch (filtered)",
)
plt.tight_layout()
plt.show()

sc.pl.scatter(adata_cr_filtered, basis="isomap", color=["potential"], size=15)

## TF Enrichment

In [ ]:
trrust_path = os.path.join(DATA_DIR, "trrust_rawdata.human.tsv")

stories.tools.tf_enrich(adata_nr_filtered, trrust_path=trrust_path)
stories.tools.tf_enrich(adata_cr_filtered, trrust_path=trrust_path)

In [ ]:
stories.tools.tf_enrich(adata_stories_nr, trrust_path=trrust_path)
stories.tools.tf_enrich(adata_stories_cr, trrust_path=trrust_path)

### Single Gene Trends

In [ ]:
stories.tools.plot_single_gene_trend(
    adata_stories_nr, "THBS1",
    s=5, alpha=0.75, annotation_key="Timepoint",
)
stories.tools.plot_single_gene_trend(
    adata_stories_cr, "THBS1",
    s=5, alpha=0.75, annotation_key="Timepoint",
)

In [ ]:
stories.tools.plot_single_gene_trend(
    adata_stories_nr, "WNT5A",
    s=5, alpha=0.75, annotation_key="Timepoint",
)
stories.tools.plot_single_gene_trend(
    adata_stories_cr, "WNT5A",
    s=5, alpha=0.75, annotation_key="Timepoint",
)

In [ ]:
sc.pl.embedding(
    adata,
    basis="X_isomap",
    color=["POSTN", "MMP14", "TNC", "THBS1"],
    vmax="p98",
    show=False,
)

fig = plt.gcf()
for ax in fig.get_axes():
    ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.show()

In [ ]:
genes = ["POSTN", "RAC1"]

conds = [
    ("NR", adata_stories_nr),
    ("CR", adata_stories_cr),
]

tp_palette = {
    "Pre": "tab:blue",
    "Resection": "tab:orange",
}

# Compute y-axis limits per gene
gene_y_lims = {}
for gene in genes:
    y_all = []
    for _, ad_cond in conds:
        y = ad_cond[:, gene].X
        if sparse.issparse(y):
            y = y.toarray()
        y_all.append(y.ravel())
    y_all = np.concatenate(y_all)
    ymin = np.nanmin(y_all)
    ymax = np.nanmax(y_all)
    margin = 0.05 * (ymax - ymin)
    gene_y_lims[gene] = (ymin - margin, ymax + margin)

fig, axes = plt.subplots(
    nrows=len(genes), ncols=len(conds), figsize=(7, 7), sharey=False,
)

if len(genes) == 1:
    axes = np.expand_dims(axes, axis=0)
if len(conds) == 1:
    axes = np.expand_dims(axes, axis=1)

for i, gene in enumerate(genes):
    ymin, ymax = gene_y_lims[gene]

    for j, (cond_label, ad_cond) in enumerate(conds):
        ax = axes[i, j]
        x = ad_cond.obs["potential"].values
        y = ad_cond[:, gene].X
        if sparse.issparse(y):
            y = y.toarray()
        y = y.ravel()

        for tp in ad_cond.obs["Timepoint"].unique():
            mask = ad_cond.obs["Timepoint"] == tp
            ax.scatter(
                x[mask], y[mask],
                s=5, alpha=0.5,
                color=tp_palette.get(tp, "gray"),
                label=tp if i == 0 and j == 0 else None,
            )

        ax.set_title(f"{gene} - {cond_label}", fontsize=14)
        ax.set_xlabel("STORIES potential", fontsize=12)
        ax.set_ylim(ymin, ymax)
        ax.tick_params(axis="both", labelsize=10)

        if j == 0:
            ax.set_ylabel("Expression", fontsize=12)

plt.tight_layout()
plt.show()